# AI Predictive Maintenance System - Phase 1 EDA & Experimentation

This notebook provides an interactive environment for:
1. **Exploratory Data Analysis (EDA)** - Understanding the dataset
2. **Feature Analysis** - Examining feature distributions and correlations
3. **Model Experimentation** - Testing different hyperparameters
4. **Results Validation** - Verifying pipeline outputs

## Dataset
- **Source**: AI4I 2020 Predictive Maintenance Dataset (UCI ML Repository)
- **Task**: Binary classification - predict machine failures
- **Target**: Target (0 = No Failure, 1 = Failure)

## Workflow
This notebook follows the ML pipeline but allows for interactive exploration and parameter tuning.

In [ ]:
# Import Required Libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path for imports
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
import xgboost as xgb
import shap

# Import preprocessing functions
from src.preprocess import (
    load_dataset, inspect_data, clean_data, feature_engineering,
    prepare_train_test, save_preprocessing_objects
)
from src.train import train_model, evaluate_model, plot_results, explain_with_shap

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful!")

In [ ]:
# Load and Inspect Dataset
print("Loading AI4I 2020 Dataset...")
df = load_dataset("data/ai4i2020.csv")
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Detailed Exploratory Data Analysis
inspect_data(df)

In [ ]:
# Visualize Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_counts = df['Target'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(target_counts.index, target_counts.values, color=colors)
axes[0].set_title('Target Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Target (0=No Failure, 1=Failure)')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Percentage plot
target_pct = df['Target'].value_counts(normalize=True) * 100
axes[1].pie(target_pct.values, labels=[f'No Failure\n({target_pct[0]:.1f}%)', 
                                        f'Failure\n({target_pct[1]:.1f}%)'],
           colors=colors, autopct='', startangle=90)
axes[1].set_title('Target Distribution (Percentage)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

imbalance_ratio = target_counts.min() / target_counts.max()
print(f"\nClass Imbalance Ratio: 1:{1/imbalance_ratio:.2f}")

In [ ]:
# Analyze Numerical Features
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'Target' in numerical_cols:
    numerical_cols.remove('Target')

print("Numerical Features Statistics:")
print(df[numerical_cols].describe())

# Correlation with Target
print("\nCorrelation with Target:")
correlations = df[numerical_cols + ['Target']].corr()['Target'].drop('Target').sort_values(ascending=False)
print(correlations)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df[numerical_cols + ['Target']].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Distributions by Target Class
fig, axes = plt.subplots(len(numerical_cols), 1, figsize=(12, 3*len(numerical_cols)))
if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    df[df['Target'] == 0][col].hist(bins=30, alpha=0.6, label='No Failure', ax=axes[idx])
    df[df['Target'] == 1][col].hist(bins=30, alpha=0.6, label='Failure', ax=axes[idx])
    axes[idx].set_title(f'Distribution: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Data Cleaning
print("Cleaning data...")
df_clean = clean_data(df)
print(f"Clean dataset shape: {df_clean.shape}")

In [ ]:
# Feature Engineering
print("Performing feature engineering...")
df_engineered, preprocessing_objects = feature_engineering(df_clean)
print(f"\nEngineered dataset shape: {df_engineered.shape}")
print(f"Preprocessing objects keys: {preprocessing_objects.keys()}")
print(f"First 5 rows of engineered data:")
print(df_engineered.head())

In [ ]:
# Train/Test Split with SMOTE
print("Preparing train/test split...")
X_train, X_test, y_train, y_test = prepare_train_test(df_engineered, apply_smote=True)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"y_train type: {type(y_train)}, shape: {y_train.shape}")
print(f"y_test type: {type(y_test)}, length: {len(y_test)}")

In [ ]:
# Train XGBoost Model
print("Training XGBoost model...")
model = train_model(X_train, y_train, n_estimators=100, max_depth=5, learning_rate=0.1)
print(f"✓ Model trained! Type: {type(model)}")

In [ ]:
# Model Evaluation
print("Evaluating model on test set...")
metrics = evaluate_model(model, X_test, y_test)

print("\n📊 Summary:")
print(f"  Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1-Score:  {metrics['f1']:.4f}")
print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")

In [ ]:
# Generate Visualizations
print("Generating visualizations...")
plot_results(model, metrics, y_test)
plt.show()

In [ ]:
# Detailed Feature Importance Analysis
feature_importance = model.feature_importances_
feature_names = [f"Feature_{i}" for i in range(len(feature_importance))]
top_indices = np.argsort(feature_importance)[-15:][::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_indices)), feature_importance[top_indices])
plt.yticks(range(len(top_indices)), [feature_names[i] for i in top_indices])
plt.xlabel('Importance Score')
plt.title('Top 15 Feature Importance', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 15 Features:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank}. {feature_names[idx]}: {feature_importance[idx]:.4f}")

In [ ]:
# SHAP Analysis (with sample data for speed)
print("Performing SHAP analysis (sample data)...")

# Sample data for SHAP visualization (full dataset can be slow)
sample_size = min(1000, len(X_test))
X_test_sample = X_test[:sample_size]

print(f"Creating SHAP explainer for {sample_size} samples...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)

# Handle binary classification output
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

print("✓ SHAP values computed!")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_class1, X_test_sample, show=False, plot_type='bar')
plt.title('SHAP Summary Plot - Mean Absolute Impact', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Scatter Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_class1, X_test_sample, show=False)
plt.title('SHAP Summary Plot - Feature Value Impact', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Example Predictions
print("Example Predictions on Test Set:\n")
y_pred = model.predict(X_test[:5])
y_pred_proba = model.predict_proba(X_test[:5])

for i in range(5):
    pred_label = "⚠️ FAILURE" if y_pred[i] == 1 else "✓ OK"
    prob_failure = y_pred_proba[i, 1]
    print(f"Sample {i+1}:")
    print(f"  Prediction: {pred_label}")
    print(f"  Probability of Failure: {prob_failure:.4f}")
    print(f"  Probability of No Failure: {1-prob_failure:.4f}")
    print()

## Hyperparameter Tuning

You can experiment with different XGBoost hyperparameters below:

- `n_estimators`: Number of boosting rounds (100-300)
- `max_depth`: Maximum tree depth (3-10)
- `learning_rate`: Learning rate (0.01-0.3)
- `subsample`: Sample rate for each tree (0.5-1.0)
- `colsample_bytree`: Feature sample rate (0.5-1.0)

Modify the training cell below to test different configurations.

In [ ]:
# Hyperparameter Tuning Experiment
# Modify these parameters to experiment:

params_to_test = [
    {'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05},
]

results = []

for i, params in enumerate(params_to_test, 1):
    print(f"\n{'='*60}")
    print(f"Configuration {i}: {params}")
    print(f"{'='*60}")
    
    model_tuned = train_model(X_train, y_train, **params)
    metrics_tuned = evaluate_model(model_tuned, X_test, y_test)
    
    results.append({
        'Config': f"Config {i}",
        'Params': params,
        'Accuracy': metrics_tuned['accuracy'],
        'Precision': metrics_tuned['precision'],
        'Recall': metrics_tuned['recall'],
        'F1': metrics_tuned['f1'],
        'ROC-AUC': metrics_tuned['roc_auc']
    })

# Compare results
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("HYPERPARAMETER COMPARISON")
print("="*60)
print(results_df.to_string(index=False))

In [ ]:
# Save the Final Model
print("Saving model and preprocessing objects...")
from src.train import save_model

save_model(model)
save_preprocessing_objects(preprocessing_objects)

print("✓ Model and preprocessing objects saved!")
print("\nLocation:")
print("  - Model: models/predictive_maintenance_model.pkl")
print("  - Preprocessing: models/preprocessing_objects.pkl")

## Summary

✓ **Phase 1 Complete!**

### Deliverables
- ✅ XGBoost machine failure prediction model
- ✅ Comprehensive evaluation metrics
- ✅ Feature importance analysis
- ✅ SHAP explainability
- ✅ Visualizations (Confusion Matrix, ROC Curve, Feature Importance)
- ✅ Saved model for inference

### Key Metrics
- Accuracy: Overall correctness of predictions
- Precision: Fraction of predicted failures that are correct
- Recall: Fraction of actual failures that are detected
- F1-Score: Harmonic mean balancing Precision and Recall
- ROC-AUC: Area under ROC curve (1.0 = perfect, 0.5 = random)

### Next Steps (Phase 2+)
- [ ] Build FastAPI backend for inference
- [ ] Create React dashboard for monitoring
- [ ] Integrate PostgreSQL database
- [ ] Add IoT sensor data streaming
- [ ] Implement model drift detection
- [ ] Deploy to production